In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.preprocessing import MinMaxScaler

path = './data/Menstrual_cramps.xlsx'

node_df = pd.read_excel(path, sheet_name='경혈단독_count')
node_df['경혈명'] = node_df['경혈명'].astype(str).str.strip().str.replace('Kl', 'KI', regex=False)

invalid_nodes = ['non-acupoint', 'ashi-point', 'BV', 'CV', 'FV']
all_nodes_list = sorted(list(set(node_df[~node_df['경혈명'].isin(invalid_nodes)]['경혈명'].tolist())))
N = len(all_nodes_list)
print(f"최종 분석 대상 노드 개수: {N}개 (5개 특수 노드 제외 완료)")

edge_df = pd.read_excel(path, sheet_name='경혈pair_count')
for col in ['Source', 'Target']:
    edge_df[col] = edge_df[col].astype(str).str.strip().str.replace('Kl', 'KI', regex=False)

# all_nodes_list에 있는 유효한 노드들 간의 연결만 남김
edge_df = edge_df[edge_df['Source'].isin(all_nodes_list) & edge_df['Target'].isin(all_nodes_list)]

G = nx.Graph()
G.add_nodes_from(all_nodes_list)
for _, row in edge_df.iterrows():
    G.add_edge(row['Source'], row['Target'], weight=row['Weight'])

print(f"그래프 G 생성 완료: 노드 {G.number_of_nodes()}개, 엣지 {G.number_of_edges()}개\n")


# 2. df_norm 생성: 그래프 G로부터 구조적 특징 추출

# 1. 중심성 지표 미리 계산 (효율성)
betweenness_dict = nx.betweenness_centrality(G)
closeness_dict = nx.closeness_centrality(G)
pagerank_dict = nx.pagerank(G, max_iter=1000)

# Eigenvector Centrality (수렴 실패 시 대안 사용)
try:
    eigenvector_dict = nx.eigenvector_centrality(G, max_iter=1000)
    print("  - Eigenvector Centrality 계산 성공")
except:
    print("  - Eigenvector Centrality 수렴 실패 -> PageRank로 대체")
    eigenvector_dict = pagerank_dict  # 수렴 실패 시 PageRank로 대체

# 2. 노드별 구조적 특징 계산
node_features = {}
for node in G.nodes():
    node_features[node] = {
        'Degree': G.degree(node),                          # 연결성
        'Clustering_Coeff': nx.clustering(G, node),        # 군집화 계수
        'Betweenness': betweenness_dict[node],             # 매개 중심성
        'Closeness': closeness_dict[node],                 # 근접 중심성
        'PageRank': pagerank_dict[node],                   # 페이지랭크
        'Eig_Centrality': eigenvector_dict[node]           # 고유벡터 중심성
    }

# 3. DataFrame 변환
df_features = pd.DataFrame.from_dict(node_features, orient='index')

# 4. 데이터 정규화 (Min-Max Scaling)
scaler = MinMaxScaler()
df_norm_values = scaler.fit_transform(df_features)
df_norm = pd.DataFrame(df_norm_values, index=df_features.index, columns=df_features.columns)

print(f"\n df_norm 생성 완료: {df_norm.shape} (노드 수 x 특징 수)")
print(f"  포함된 특징: {list(df_norm.columns)}")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import numpy as np

print("=== RolX 최적 클러스터 수(K) 탐색 ===\n")

k_range = range(2, 11)
inertias = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = km.fit_predict(df_norm)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(df_norm, labels))

# 시각화
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.set_xlabel('Number of Roles (K)', fontsize=12)
ax1.set_ylabel('Inertia (SSE)', color='tab:blue', fontsize=12)
ax1.plot(k_range, inertias, marker='o', color='tab:blue', lw=2, label='Inertia')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.set_ylabel('Silhouette Score', color='tab:red', fontsize=12)
ax2.plot(k_range, sil_scores, marker='s', color='tab:red', lw=2, label='Silhouette Score')
ax2.tick_params(axis='y', labelcolor='tab:red')

plt.title('Selecting Optimal K for RolX (Elbow & Silhouette)', fontsize=15, fontweight='bold')
fig.tight_layout()
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig("RolX_K_Selection.png", dpi=300, bbox_inches='tight')
plt.show()

best_k_sil = k_range[np.argmax(sil_scores)]
print(f"\n[분석 결과] 실루엣 점수 기준 최적 K: {best_k_sil}")

# K 값 출력
print("\nK | Silhouette Score")
print("-" * 25)
for k, sil in zip(k_range, sil_scores):
    marker = " ← 최적" if k == best_k_sil else ""
    print(f"{k:>1} | {sil:.4f}{marker}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import networkx as nx
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

K_final = best_k_sil

print(f"선택된 K: {K_final}")
km_final = KMeans(n_clusters=K_final, n_init=20, random_state=42)
role_labels_final = km_final.fit_predict(df_norm)

names = df_norm.index.tolist()
degree_dict = dict(G.degree())


role_info = {}
role_groups = {}
partition_role = {name: role_labels_final[i] for i, name in enumerate(names)}

for k in range(K_final):
    indices = np.where(role_labels_final == k)[0]
    members = [names[i] for i in indices]
    leader = max(members, key=lambda x: degree_dict.get(x, 0))
    role_info[k] = leader
    role_groups[k] = members

tsne = TSNE(n_components=2, perplexity=15, learning_rate='auto', init='pca', random_state=42)
X_tsne = tsne.fit_transform(df_norm)
pos_tsne = {n: X_tsne[i] for i, n in enumerate(names)}

node_sizes = []
for i, name in enumerate(names):
    gid = role_labels_final[i]
    if name == role_info[gid]:
        node_sizes.append(1300)  # 리더 (크게)
    else:
        node_sizes.append(500)   # 일반 노드 (작게)

plt.figure(figsize=(18, 14))

if K_final == 2:
    colors = ['#1f77b4', '#ff7f0e']
    cmap = lambda x: colors[x]
elif K_final <= 10:
    cmap = cm.get_cmap('tab10')
else:
    cmap = cm.get_cmap('tab20')

node_colors = [cmap(role_labels_final[i]) for i in range(len(names))]
nx.draw_networkx_nodes(G, pos=pos_tsne,
                       node_color=node_colors,
                       node_size=node_sizes,
                       alpha=0.9,
                       edgecolors='white',
                       linewidths=1.5)

# 라벨링
nx.draw_networkx_labels(G, pos_tsne, font_size=9, font_family='sans-serif', font_weight='bold')

legend_handles = []
for i in sorted(role_info.keys()):
    group_color = cmap(i) if K_final > 2 else colors[i]
    leader_name = role_info[i]
    count = len(role_groups[i])

    label_text = f"Role {i+1} (Leader: {leader_name}, n={count})"
    patch = mpatches.Patch(color=group_color, label=label_text)
    legend_handles.append(patch)

plt.legend(handles=legend_handles,
           title="Role Info (Leader & Size)",
           loc='upper left',
           bbox_to_anchor=(1, 1),
           fontsize=11,
           title_fontsize=13,
           frameon=True,
           shadow=True)

plt.title(f"RolX Structural Role Clustering (t-SNE Projection, K={K_final})",
          fontsize=22, fontweight='bold', pad=20)
plt.axis('off')
plt.tight_layout()

plt.savefig(f"RolX_tSNE_K{K_final}.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print(f"=== RolX 구조적 역할 분석 상세 결과 (총 {K_final}개 역할) ===\n")

for i in sorted(role_groups.keys()):
    leader_name = role_info[i]    # 위 시각화 코드에서 정의된 리더 (Degree 1위)
    members = role_groups[i]      # 위 시각화 코드에서 정의된 멤버 리스트
    count = len(members)

    sorted_members = sorted(members)

    print(f" Role {i+1}")
    print(f"  - 대표 경혈(Leader): {leader_name}")
    print(f"  - 소속 경혈 수: {count}개")
    print(f"  - 경혈 리스트: {', '.join(sorted_members)}")
    print("-" * 50)

import pandas as pd

rolx_summary_data = []
for i in sorted(role_groups.keys()):
    rolx_summary_data.append({
        'Role': f"Role {i+1}",
        'Leader': role_info[i],
        'Count': len(role_groups[i]),
        'Members': ', '.join(sorted(role_groups[i]))
    })

df_rolx_summary = pd.DataFrame(rolx_summary_data)